# 01 注意力机制基础：从缩放点积到 MHA/MQA/GQA

## 简介

本笔记本介绍大语言模型最核心的组件——注意力机制（Attention）。
我们将从最基本的缩放点积注意力公式开始，逐步构建到多头注意力（MHA）、分组查询注意力（GQA）和多查询注意力（MQA），
并比较它们的参数量和 KV Cache 内存占用。

## 1. 导入

In [ ]:
import torch
import torch.nn.functional as F
from models.common.attention import MultiHeadAttention, GroupedQueryAttention

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

### 🧠 直觉理解：注意力机制

注意力机制就像在一个图书馆里找书——你带着一个**查询问题**（Query），每本书的**书名标签**（Key）告诉你它是否相关，而书的**实际内容**（Value）才是你真正需要的。 注意力机制的核心就是：用 Query 和 Key 的相似度来决定从每个 Value 中提取多少信息。

**类比**：想象你在一场聚会上，你的注意力（Query）会自然地被与你兴趣相关的话题（Key）吸引，然后你会从那些话题中获取有价值的信息（Value）。注意力权重就是你对每个话题的"关注程度"，越相关的话题权重越高。缩放因子 √d_k 则像是聚会上控制音量的旋钮——防止所有人同时大声说话（点积过大）导致你听不清任何一个人（softmax 梯度消失）。

### 📐 数学原理：缩放点积注意力

注意力函数将一个查询和一组键值对映射到一个输出，核心公式为：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

展开各步骤：

1. **相似度计算**：$S = QK^T \in \mathbb{R}^{n \times n}$，每个元素 $S_{ij} = q_i \cdot k_j$

2. **缩放**：$\hat{S} = S / \sqrt{d_k}$，防止当 $d_k$ 较大时点积值过大

3. **Softmax 归一化**：$\alpha_{ij} = \frac{e^{\hat{S}_{ij}}}{\sum_k e^{\hat{S}_{ik}}}$，每行和为 1

4. **加权求和**：$o_i = \sum_j \alpha_{ij} v_j$

**为什么需要缩放？** 当 $d_k$ 较大时，$q \cdot k$ 的方差为 $d_k$（假设各分量独立），导致 softmax 输入值域过大，梯度趋近于 0。除以 $\sqrt{d_k}$ 使方差恢复为 1。

### 🧠 直觉理解：注意力机制

注意力机制就像在一个图书馆里找书——你带着一个**查询问题**（Query），每本书的**书名标签**（Key）告诉你它是否相关，而书的**实际内容**（Value）才是你真正需要的。 注意力机制的核心就是：用 Query 和 Key 的相似度来决定从每个 Value 中提取多少信息。

**类比**：想象你在一场聚会上，你的注意力（Query）会自然地被与你兴趣相关的话题（Key）吸引，然后你会从那些话题中获取有价值的信息（Value）。注意力权重就是你对每个话题的"关注程度"，越相关的话题权重越高。缩放因子 √d_k 则像是聚会上控制音量的旋钮——防止所有人同时大声说话（点积过大）导致你听不清任何一个人（softmax 梯度消失）。

### 📐 数学原理：缩放点积注意力

注意力函数将一个查询和一组键值对映射到一个输出，核心公式为：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

展开各步骤：

1. **相似度计算**：$S = QK^T \in \mathbb{R}^{n \times n}$，每个元素 $S_{ij} = q_i \cdot k_j$

2. **缩放**：$\hat{S} = S / \sqrt{d_k}$，防止当 $d_k$ 较大时点积值过大

3. **Softmax 归一化**：$\alpha_{ij} = \frac{e^{\hat{S}_{ij}}}{\sum_k e^{\hat{S}_{ik}}}$，每行和为 1

4. **加权求和**：$o_i = \sum_j \alpha_{ij} v_j$

**为什么需要缩放？** 当 $d_k$ 较大时，$q \cdot k$ 的方差为 $d_k$（假设各分量独立），导致 softmax 输入值域过大，梯度趋近于 0。除以 $\sqrt{d_k}$ 使方差恢复为 1。

## 2. 缩放点积注意力（Scaled Dot-Product Attention）

注意力机制的核心公式：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中：
- **Q (Query)**: 查询向量，表示"我想查找什么"
- **K (Key)**: 键向量，表示"我能提供什么"
- **V (Value)**: 值向量，表示"我的实际内容"
- **√(d_k)**: 缩放因子，防止点积过大导致 softmax 梯度消失

让我们手动步骤步实现一次缩放点积注意力计算：

In [ ]:
# 步骤 1: 创建模拟的 Q, K, V 张量
B, S, D, n_heads = 2, 4, 64, 8
head_dim = D // n_heads  # 每个头的维度 = 8

# 模拟一个头的 Q, K, V
torch.manual_seed(42)
q = torch.randn(B, n_heads, S, head_dim)  # (2, 8, 4, 8)
k = torch.randn(B, n_heads, S, head_dim)
v = torch.randn(B, n_heads, S, head_dim)

print(f"Q 形状: {list(q.shape)}")
print(f"K 形状: {list(k.shape)}")
print(f"V 形状: {list(v.shape)}")

In [ ]:
# 步骤 2: 计算注意力得分 Q @ K^T / sqrt(d_k)
scale = head_dim ** 0.5  # sqrt(8) = 2.828
scores = (q @ k.transpose(-2, -1)) / scale  # (2, 8, 4, 4)
print(f"注意力得分形状: {list(scores.shape)}")
print(f"得分矩阵 (第一个 batch, 第一个头):")
print(scores[0, 0])

In [ ]:
# 步骤 3: 施加因果掩码 + Softmax
# 因果掩码：上三角为 True（将被mask为 -inf）
causal_mask = torch.triu(
    torch.ones(S, S, dtype=torch.bool), diagonal=1
).unsqueeze(0).unsqueeze(0)  # (1, 1, 4, 4)

print("因果掩码 (True=被屏蔽):")
for i in range(S):
    row = "".join("X" if causal_mask[0,0,i,j] else "O" for j in range(S))
    print(f"  token[{i}] -> {row}  (O=可见, X=被mask)")

scores_masked = scores.masked_fill(causal_mask, float('-inf'))
attn_weights = torch.softmax(scores_masked, dim=-1)
print(f"\n注意力权重形状: {list(attn_weights.shape)}")
print(f"权重矩阵(每行和=1):")
print(attn_weights[0, 0])

In [ ]:
# 步骤 4: 加权求和 attn_weights @ V
output = attn_weights @ v  # (2, 8, 4, 8)
print(f"输出形状: {list(output.shape)}")
print(f"第一个头的输出 (token 0): {output[0, 0, 0, :4]}")

### 🧠 直觉理解：MHA / GQA / MQA

想象一个公司会议：MHA 就像每个部门经理（Q头）都有自己的秘书（KV头），信息最完整但人力成本最高；MQA 就像所有部门经理共享一个秘书，效率最高但信息可能不够精细；GQA 则是折中方案——几个相关的部门经理共享一个秘书，在效率和效果之间取得平衡。

**关键洞察**：推理时 KV Cache 的大小与 KV 头数成正比。减少 KV 头数可以显著降低推理显存，让更大的 batch size 成为可能，从而提升吞吐量。GQA 是 LLaMA2/3 的默认选择，因为它在质量损失很小的情况下大幅减少了 KV Cache。

### 📐 数学原理：KV Cache 内存计算

对于 MHA，每个 token 需要存储的 KV Cache 大小为：

$$\text{KV Cache per token} = 2 \times n_{kv\_heads} \times d_{head} \times \text{dtype\_bytes}$$

对于整个序列（$S$ 个 token）和 $L$ 层：

$$\text{Total KV Cache} = 2 \times B \times L \times n_{kv\_heads} \times S \times d_{head} \times \text{dtype\_bytes}$$

| 类型 | $n_{kv\_heads}$ | 相对 MHA 的 KV Cache |
|---|---|---|
| MHA | $n_h$ | 100% |
| GQA | $n_h / g$ | $1/g$ |
| MQA | 1 | $1/n_h$ |

其中 $g$ 为 GQA 的分组数（每组 Q 头共享 1 个 KV 头）。

### 🧠 直觉理解：MHA / GQA / MQA

想象一个公司会议：MHA 就像每个部门经理（Q头）都有自己的秘书（KV头），信息最完整但人力成本最高；MQA 就像所有部门经理共享一个秘书，效率最高但信息可能不够精细；GQA 则是折中方案——几个相关的部门经理共享一个秘书，在效率和效果之间取得平衡。

**关键洞察**：推理时 KV Cache 的大小与 KV 头数成正比。减少 KV 头数可以显著降低推理显存，让更大的 batch size 成为可能，从而提升吞吐量。GQA 是 LLaMA2/3 的默认选择，因为它在质量损失很小的情况下大幅减少了 KV Cache。

### 📐 数学原理：KV Cache 内存计算

对于 MHA，每个 token 需要存储的 KV Cache 大小为：

$$\text{KV Cache per token} = 2 \times n_{kv\_heads} \times d_{head} \times \text{dtype\_bytes}$$

对于整个序列（$S$ 个 token）和 $L$ 层：

$$\text{Total KV Cache} = 2 \times B \times L \times n_{kv\_heads} \times S \times d_{head} \times \text{dtype\_bytes}$$

| 类型 | $n_{kv\_heads}$ | 相对 MHA 的 KV Cache |
|---|---|---|
| MHA | $n_h$ | 100% |
| GQA | $n_h / g$ | $1/g$ |
| MQA | 1 | $1/n_h$ |

其中 $g$ 为 GQA 的分组数（每组 Q 头共享 1 个 KV 头）。

## 3. 多头注意力 vs 分组查询 vs 多查询

| 类型 | Q 头数 | KV 头数 | 特点 |
|---|---|---|---|
| **MHA** | n_heads | n_heads | 每个 Q 头有独立的 KV，计算量最大|
| **GQA** | n_heads | n_kv_heads < n_heads | KV 头共享，平衡速度和质量|
| **MQA** | n_heads | 1 | 所有 Q 头共享 1 个 KV，KV Cache 最小|

我们的实现中，`GroupedQueryAttention` 当 `n_kv_heads==n_heads` 时等价于 MHA，当 `n_kv_heads==1` 时等价于 MQA。

In [ ]:
# 实例化三种注意力机制
dim, n_heads, n_kv_heads = 64, 8, 2

mha = MultiHeadAttention(dim=dim, n_heads=n_heads)
gqa = GroupedQueryAttention(dim=dim, n_heads=n_heads, n_kv_heads=n_kv_heads)
mqa = GroupedQueryAttention(dim=dim, n_heads=n_heads, n_kv_heads=1)

# 统计参数量
def count_params(model, name):
    n = sum(p.numel() for p in model.parameters())
    print(f"{name}: {n:,} 参数")
    return n

p_mha = count_params(mha, "MHA")
p_gqa = count_params(gqa, "GQA")
p_mqa = count_params(mqa, "MQA")
print(f"\nGQA 参数为 MHA 的 {p_gqa/p_mha*100:.1f}%")
print(f"MQA 参数为 MHA 的 {p_mqa/p_mha*100:.1f}%")

## 4. 实际运行验证

In [ ]:
# 前向传播并输出各步张量形状
x = torch.randn(2, 16, 64)  # (batch=2, seq=16, dim=64)

print("=" * 50)
print("MHA 前向传播：")
print("=" * 50)
mha = MultiHeadAttention(dim=64, n_heads=8)
out_mha = mha(x, use_causal_mask=True)
print(f"输入:  {list(x.shape)}")
print(f"输出: {list(out_mha.shape)}")
print(f"形状一致: {out_mha.shape == x.shape}")

print("\n" + "=" * 50)
print("MQA 前向传播 (n_kv_heads=1)：")
print("=" * 50)
mqa = GroupedQueryAttention(dim=64, n_heads=8, n_kv_heads=1)
out_mqa = mqa(x, use_causal_mask=True)
print(f"输入:  {list(x.shape)}")
print(f"输出: {list(out_mqa.shape)}")
print(f"形状一致: {out_mqa.shape == x.shape}")

## 5. KV Cache 内存占用对比

KV Cache 是推理时存储已计算的 Key 和 Value，避免重复计算。
其内存占用与 KV 头数成正比。

In [ ]:
def kv_cache_size(n_heads, n_kv_heads, head_dim, seq_len, batch_size=1, dtype_bytes=2):
    """计算 KV Cache 内存占用 (MB)"""
    # 每层需存 K + V
    k_bytes = batch_size * n_kv_heads * seq_len * head_dim * dtype_bytes
    v_bytes = batch_size * n_kv_heads * seq_len * head_dim * dtype_bytes
    return (k_bytes + v_bytes) / (1024 ** 2)

seq_len = 4096
head_dim = 64
batch_size = 1

# 对比 MHA(8头), GQA(8头Q,2头KV), MQA(8头Q,1头KV)
for n_q_heads, n_kv_heads, name in [(8, 8, "MHA"), (8, 2, "GQA"), (8, 1, "MQA")]:
    size_mb = kv_cache_size(n_q_heads, n_kv_heads, head_dim, seq_len, batch_size)
    print(f"{name}: KV heads={n_kv_heads}, Cache内存={size_mb:.2f} MB (每层)")

# 如果是 32 层的 LLaMA3-8B 级别模型
n_layers = 32
print(f"\n× {n_layers} 层总占用:")
for n_q_heads, n_kv_heads, name in [(8, 8, "MHA"), (8, 2, "GQA"), (8, 1, "MQA")]:
    size_mb = kv_cache_size(n_q_heads, n_kv_heads, head_dim, seq_len, batch_size) * n_layers
    print(f"  {name}: {size_mb:.0f} MB")

### 📊 KV Cache 内存对比柱状图

用柱状图直观对比 MHA、GQA、MQA 三种注意力机制的 KV Cache 内存占用。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# KV Cache 对比参数
seq_len = 4096
head_dim = 64
n_layers = 32
dtype_bytes = 2  # fp16

configs = [
    ('MHA\n(8 KV heads)', 8),
    ('GQA\n(2 KV heads)', 2),
    ('MQA\n(1 KV head)', 1),
]

labels = [c[0] for c in configs]
kv_heads = [c[1] for c in configs]
sizes_mb = [2 * kv_h * head_dim * seq_len * dtype_bytes * n_layers / (1024**2) for kv_h in kv_heads]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax.bar(labels, sizes_mb, color=colors, width=0.5, edgecolor='black', linewidth=0.5)

for bar, size in zip(bars, sizes_mb):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            f'{size:.0f} MB', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('KV Cache 总内存 (MB)', fontsize=12)
ax.set_title(f'KV Cache 内存对比 (seq_len={seq_len}, {n_layers}层, fp16)', fontsize=14)
ax.set_ylim(0, max(sizes_mb) * 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 📊 KV Cache 内存对比柱状图

用柱状图直观对比 MHA、GQA、MQA 三种注意力机制的 KV Cache 内存占用。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# KV Cache 对比参数
seq_len = 4096
head_dim = 64
n_layers = 32
dtype_bytes = 2  # fp16

configs = [
    ('MHA\n(8 KV heads)', 8),
    ('GQA\n(2 KV heads)', 2),
    ('MQA\n(1 KV head)', 1),
]

labels = [c[0] for c in configs]
kv_heads = [c[1] for c in configs]
sizes_mb = [2 * kv_h * head_dim * seq_len * dtype_bytes * n_layers / (1024**2) for kv_h in kv_heads]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax.bar(labels, sizes_mb, color=colors, width=0.5, edgecolor='black', linewidth=0.5)

for bar, size in zip(bars, sizes_mb):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
            f'{size:.0f} MB', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('KV Cache 总内存 (MB)', fontsize=12)
ax.set_title(f'KV Cache 内存对比 (seq_len={seq_len}, {n_layers}层, fp16)', fontsize=14)
ax.set_ylim(0, max(sizes_mb) * 1.2)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 注意力得分可视化

使用 ASCII 图展示注意力权重矩阵和因果掩码的效果。

In [ ]:
import math

# 模拟 8 个 token 的注意力分布
torch.manual_seed(0)
S_demo = 8
q_demo = torch.randn(1, 1, S_demo, 4)
k_demo = torch.randn(1, 1, S_demo, 4)
scores_demo = (q_demo @ k_demo.transpose(-2, -1)) / math.sqrt(4)
mask_demo = torch.triu(torch.ones(S_demo, S_demo, dtype=torch.bool), diagonal=1)
scores_masked_demo = scores_demo.masked_fill(mask_demo, float('-inf'))
attn_demo = torch.softmax(scores_masked_demo, dim=-1)[0, 0]

print("注意力权重热力图 (ASCII):")
print("  每行为一个 query token 对所有 key token 的注意力分布")
print("  '░'=低权重, '█'=高权重, '.'=mask掉(0)")
print()
chars = " .=os#@"
max_val = attn_demo.max().item()
print("   " + "".join(f" K{i}" for i in range(S_demo)))
for i in range(S_demo):
    row = f"Q{i} "
    for j in range(S_demo):
        if mask_demo[i, j]:
            row += " · "  # 被mask
        else:
            v = attn_demo[i, j].item() / max_val
            idx = min(int(v * (len(chars) - 1)), len(chars) - 1)
            row += f" {chars[idx]} "
    print(row)
print()
print("箭头方向:每个 Q token 只能看到自己和之前的 K/V token")

### 📊 注意力权重热力图可视化

下面用 matplotlib 将注意力权重矩阵可视化为热力图，更直观地展示因果掩码和注意力分布。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 注意力权重热力图
attn_np = attn_demo.numpy()
im = axes[0].imshow(attn_np, cmap='YlOrRd', aspect='auto')
axes[0].set_title('注意力权重热力图', fontsize=14)
axes[0].set_xlabel('Key Token 位置')
axes[0].set_ylabel('Query Token 位置')
axes[0].set_xticks(range(S_demo))
axes[0].set_yticks(range(S_demo))
plt.colorbar(im, ax=axes[0], label='权重值')

# 右图: 因果掩码
mask_np = mask_demo.numpy().astype(float)
axes[1].imshow(mask_np, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('因果掩码 (绿色=可见, 红色=屏蔽)', fontsize=14)
axes[1].set_xlabel('Key Token 位置')
axes[1].set_ylabel('Query Token 位置')
axes[1].set_xticks(range(S_demo))
axes[1].set_yticks(range(S_demo))

plt.tight_layout()
plt.show()

### 📊 注意力权重热力图可视化

下面用 matplotlib 将注意力权重矩阵可视化为热力图，更直观地展示因果掩码和注意力分布。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: 注意力权重热力图
attn_np = attn_demo.numpy()
im = axes[0].imshow(attn_np, cmap='YlOrRd', aspect='auto')
axes[0].set_title('注意力权重热力图', fontsize=14)
axes[0].set_xlabel('Key Token 位置')
axes[0].set_ylabel('Query Token 位置')
axes[0].set_xticks(range(S_demo))
axes[0].set_yticks(range(S_demo))
plt.colorbar(im, ax=axes[0], label='权重值')

# 右图: 因果掩码
mask_np = mask_demo.numpy().astype(float)
axes[1].imshow(mask_np, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('因果掩码 (绿色=可见, 红色=屏蔽)', fontsize=14)
axes[1].set_xlabel('Key Token 位置')
axes[1].set_ylabel('Query Token 位置')
axes[1].set_xticks(range(S_demo))
axes[1].set_yticks(range(S_demo))

plt.tight_layout()
plt.show()

## 7. 关键要点总结

1. **缩放点积注意力**是所有注意力变体的基础，核心是 softmax(QK^T/√d_k) × V
2. **因果掩码**保证自回归生成时 token_i 只能看到 token_{0..i}
3. **MHA**: 每个 Q 头有独立的 KV，表达能力最强
4. **GQA**: KV 头数介于 1 和 n_heads 之间，LLaMA2/3 的默认选择
5. **MQA**: 只有 1 个 KV 头，KV Cache 最小，但质量略有下降
6. **KV Cache** 内存占用 = 2 × batch × n_kv_heads × seq_len × head_dim × dtype_bytes

## 📝 练习题

### 思考题
1. 为什么缩放因子是 $\sqrt{d_k}$ 而不是 $d_k$？如果使用 $d_k$ 作为缩放因子，注意力分布会发生什么变化？

### 编程题
2. 修改上面的代码，实现一个不带缩放的点积注意力（即去掉 $\sqrt{d_k}$），对比当 $d_k=64$ 和 $d_k=8$ 时 softmax 输出的最大值和梯度，验证缩放的必要性。

### 分析题
3. 假设一个模型有 32 层、32 个 Q 头、head_dim=128，序列长度 8192，batch_size=4。计算 MHA、GQA(kv_heads=8)、MQA 三种配置下的 KV Cache 总内存（GB），并分析在 80GB GPU 上能支持的最大 batch_size。

## 📝 练习题

### 思考题
1. 为什么缩放因子是 $\sqrt{d_k}$ 而不是 $d_k$？如果使用 $d_k$ 作为缩放因子，注意力分布会发生什么变化？

### 编程题
2. 修改上面的代码，实现一个不带缩放的点积注意力（即去掉 $\sqrt{d_k}$），对比当 $d_k=64$ 和 $d_k=8$ 时 softmax 输出的最大值和梯度，验证缩放的必要性。

### 分析题
3. 假设一个模型有 32 层、32 个 Q 头、head_dim=128，序列长度 8192，batch_size=4。计算 MHA、GQA(kv_heads=8)、MQA 三种配置下的 KV Cache 总内存（GB），并分析在 80GB GPU 上能支持的最大 batch_size。